In [1]:
!pip install datasets pandas


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
# Load Hinglish → English dataset
hing_ds = load_dataset("CodeMixBench/CodeMixBench", "mt_hineng_eng")
# Load Spanglish → English dataset
span_ds = load_dataset("CodeMixBench/CodeMixBench", "mt_spaeng_eng")


In [4]:
# Convert the 'test' split to DataFrames (it contains the full dataset)
hing_df = hing_ds["test"].to_pandas()
span_df = span_ds["test"].to_pandas()

print("Raw Hinglish sample:")
print(hing_df.head())

print("\nRaw Spanglish sample:")
print(span_df.head())


Raw Hinglish sample:
   index                                           sentence  \
0      0                                              hello   
1      1  hello yar, mein is movie ko nahi dekha hoon th...   
2      2               acha tho is movie kis baare me hein?   
3      3      is movie tho social network ke bare mein hein   
4      4                     mein aise kuch nahi dekha hoon   

                                              answer  
0                                              hello  
1  hello there, I have not seen this movie so im ...  
2           Alright that is fine. What is the movie?  
3                    The movie is The Social Network  
4                   I have not seen that one either.  

Raw Spanglish sample:
   index                                           sentence  \
0      0  Podría ser que alguna red masiva de communicat...   
1      1  Si podemos show some of the video, podrán see ...   
2      2  Es un diseño absolutamente fat-free, y cuando ..

In [5]:
# Keep only required columns and rename them
hing_df = hing_df[['sentence', 'answer']].rename(columns={
    'sentence': 'source', 
    'answer': 'target'
})

span_df = span_df[['sentence', 'answer']].rename(columns={
    'sentence': 'source', 
    'answer': 'target'
})

# Clean empty or NaN rows
hing_df = hing_df.dropna().reset_index(drop=True)
span_df = span_df.dropna().reset_index(drop=True)

hing_df = hing_df[(hing_df['source'].str.strip() != '') & (hing_df['target'].str.strip() != '')]
span_df = span_df[(span_df['source'].str.strip() != '') & (span_df['target'].str.strip() != '')]

print("Cleaned Hinglish columns:", hing_df.columns)
print("Hinglish size:", len(hing_df))

print("\nCleaned Spanglish columns:", span_df.columns)
print("Spanglish size:", len(span_df))


Cleaned Hinglish columns: Index(['source', 'target'], dtype='object')
Hinglish size: 942

Cleaned Spanglish columns: Index(['source', 'target'], dtype='object')
Spanglish size: 1059


In [6]:
def create_splits(df, train_frac=0.8, val_frac=0.1, random_state=42):
    """
    Splits df into train/val/test with fractions train_frac, val_frac, test_frac.
    test_frac is inferred as 1 - train_frac - val_frac.
    """
    assert train_frac + val_frac < 1.0, "train_frac + val_frac must be < 1"
    test_frac = 1.0 - train_frac - val_frac

    # First split off train
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - train_frac),
        random_state=random_state,
        shuffle=True,
    )

    # Now split temp into val + test with the right proportions
    relative_val_frac = val_frac / (val_frac + test_frac)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - relative_val_frac),
        random_state=random_state,
        shuffle=True,
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

# Apply splits
hing_train, hing_val, hing_test = create_splits(hing_df, train_frac=0.8, val_frac=0.1)
span_train, span_val, span_test = create_splits(span_df, train_frac=0.8, val_frac=0.1)

print("HINGLISH → train:", len(hing_train), "val:", len(hing_val), "test:", len(hing_test))
print("SPANGLISH → train:", len(span_train), "val:", len(span_val), "test:", len(span_test))


HINGLISH → train: 753 val: 94 test: 95
SPANGLISH → train: 847 val: 106 test: 106


In [7]:
# Save for reuse
hing_train.to_csv("hinglish_train.csv", index=False)
hing_val.to_csv("hinglish_val.csv", index=False)
hing_test.to_csv("hinglish_test.csv", index=False)

span_train.to_csv("spanglish_train.csv", index=False)
span_val.to_csv("spanglish_val.csv", index=False)
span_test.to_csv("spanglish_test.csv", index=False)

print("Splits saved successfully!")


Splits saved successfully!
